In [ ]:
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

In [ ]:
model = joblib.load("shipment_delay_model.pkl")

In [ ]:
print(type(model))

<class 'xgboost.sklearn.XGBClassifier'>


In [ ]:
df = pd.read_csv("../dataset/supply_chain_data.csv")

In [ ]:
df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [ ]:
df["Shipment_Delay"] = (df["Shipping times"] > 5).astype(int)

In [ ]:
df[["Shipping times", "Shipment_Delay"]].head(10)

,Shipping times,Shipment_Delay
0,4,0
1,2,0
2,2,0
3,6,1
4,8,1
5,3,0
6,8,1
7,1,0
8,7,1
9,1,0


In [ ]:
label_encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("Categorical columns encoded successfully!")

Categorical columns encoded successfully!


C:\Users\shiva\AppData\Local\Temp\ipykernel_19004\426972075.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    int64  
 1   SKU                      100 non-null    int64  
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    int64  
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    int64  
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    int64  
 14  Location                 100 non-null 

In [ ]:
new_data = df.drop("Shipment_Delay", axis=1)

In [ ]:
X = df.drop("Shipment_Delay", axis=1)

In [ ]:
predictions = model.predict(new_data)

In [ ]:
print(predictions[:10])

[0 0 0 1 1 0 1 0 1 0]


In [ ]:
new_data["Predicted_Shipment_Delay"] = predictions

In [ ]:
new_data.head(10)

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs,Predicted_Shipment_Delay
0,1,0,69.808006,55,802,8661.996792,2,58,7,96,...,29,215,29,46.279879,2,0.226410,2,1,187.752075,0
1,2,1,14.843523,95,736,7460.900065,0,53,30,37,...,23,517,30,33.616769,2,4.854068,2,1,503.065579,0
2,1,12,11.319683,34,8,9577.749626,3,1,10,88,...,12,971,27,30.688019,2,4.580593,0,2,141.920282,0
3,2,23,61.163343,68,83,7766.836426,2,23,13,59,...,24,937,18,35.624741,0,4.746649,1,0,254.776159,1
4,2,34,4.805496,26,871,2686.505152,2,5,3,56,...,5,414,3,92.065161,0,3.145580,0,0,923.440632,1
5,1,45,1.699976,87,147,2828.348746,2,90,27,66,...,10,104,17,56.766476,0,2.779194,2,0,235.461237,0
6,2,56,4.078333,48,65,7823.476560,1,11,15,58,...,14,314,24,1.085069,2,1.000911,3,0,134.369097,1
7,0,67,42.958384,59,426,8496.103813,0,93,17,11,...,22,564,1,99.466109,0,0.398177,2,2,802.056312,0
8,0,78,68.717597,78,150,7517.363211,0,5,10,15,...,13,769,8,11.423027,2,2.709863,3,1,505.557134,1
9,2,89,64.015733,35,980,4971.145988,3,14,27,83,...,29,963,23,47.957602,2,3.844614,1,1,995.929461,0


In [ ]:
new_data.to_csv("../dataset/predicted_shipments.csv", index=False)